---
title: HI-VQE Chemistry - A Qiskit Function by Qunova Computing
description: Documentation for Qunova Computing's HI-VQE Qiskit Function. It can be used to solve for the ground state and ground state energy of chemical systems.
---


{/* cspell:ignore asstr lumo hivqe DMRG ccsd, ffsim, pvdz, Qunova, HI-VQE, AVAS, Dooh, pyscf, nucmod, GTO's, lucj, pseudopotential, pseudopotentials, magmom, Jastrow, multireference, Pellow, Jarman */}

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

# HI-VQE Chemistry - A Qiskit Function by Qunova Computing

*See the [API reference](/docs/api/functions/qunova-chemistry)*

<Admonition type="note">
  Qiskit Functions are an experimental feature available only to IBM Quantum&reg; Premium Plan, Flex Plan, and On-Prem (via IBM Quantum Platform API) Plan users. They are in preview release status and subject to change.
</Admonition>

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.45.0
```
</AccordionItem>
</Accordion>

## Overview

In quantum chemistry, the electronic structure problem focuses on finding the solutions to the electronic Schrödinger equation - the quantum wave functions describing the behavior of the system's electrons. These wave functions are vectors of complex amplitudes, with each amplitude corresponding to the contribution of a possible electron configuration.

The ground state is the lowest energy wave function of the system and has a special importance in the study of molecular systems. The most accurate approach for computing the ground state considers all possible electron configurations, but this becomes intractable for larger systems since the number of configurations grows exponentially with system size.

The Handover Iterative Variational Quantum Eigensolver (HI-VQE) is an innovative hybrid quantum-classical method for accurately estimating the ground state of molecular systems. It integrates quantum hardware with classical computing, using quantum processors to efficiently explore candidate electron configurations and calculating the resulting wave function on classical computers. By generating compact yet chemically accurate wave functions, HI-VQE enhances research and discovery in quantum chemistry and materials science.

![Image showing an overview of Qunova's HI-VQE algorithm](/docs/images/guides/qunova-chemistry/overview.svg)

HI-VQE reduces the computational complexity of the electronic structure problem by efficiently estimating the ground state with high accuracy. It focuses on a carefully selected subset of the most relevant electron configurations, optimizing both accuracy and efficiency.

Combining the strengths of both classical and quantum computers, HI-VQE iteratively refines and improves the current estimate wave function. Its unique subspace construction techniques help make configuration selection more efficient, so that users have greater computational control and improved accuracy in quantum chemistry simulations.

If you would like to learn about the algorithm in more depth, you can [read the associated research paper](https://arxiv.org/abs/2503.06292)

## Description

The number of electron configurations for a molecular system grows exponentially with system size. However, for certain electronic states, such as the ground state, it is common that only a small fraction of configurations significantly contribute to the state’s energy. Selected configuration interaction (SCI) methods exploit this sparsity to reduce computational costs by identifying and focusing on the most relevant configurations. This subset of configurations is referred to as a subspace.

HI-VQE leverages the inherent efficiency of quantum computers for representing molecular systems to aid the subspace search. It integrates classical and quantum subroutines to solve the electronic structure problem with high accuracy. Unlike existing quantum SCI methods, HI-VQE combines variational training, iterative subspace construction, and pre-diagonalization configuration screening to enhance efficiency by reducing quantum measurements, iterations, and classical diagonalization costs. HI-VQE can therefore be applied to larger molecular systems which require more qubits, and reduces the cost to solve a problem of a given size to the same degree of accuracy.

![Image showing a detailed description of each step in Qunova's HI-VQE algorithm.](/docs/images/guides/qunova-chemistry/description.avif)

To compute a system's ground state, HI-VQE first uses the classical chemistry package PySCF to generate a molecular representation from user-provided inputs, such as the molecular geometry and other molecular information. It then enters a hybrid quantum-classical optimization loop, iteratively refining a subspace to optimally represent the ground state while minimizing the number of configurations included. The loop continues until convergence criteria, such as subspace size or energy stability, are met, after which the computed ground state wave function and energy are output. These results can be used to construct accurate potential energy surfaces and perform further analysis of the system.

The optimization loop focuses on adjusting a quantum circuit’s parameters to generate a high-quality subspace. HI-VQE offers three quantum circuit options: [`excitation_preserving`](/docs/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](/docs/api/qiskit/qiskit.circuit.library.efficient_su2), and [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). The optimization is initialized close to the Hartree-Fock reference state due to its general suitability. The circuit is then executed on a quantum device and configurations are sampled from the resulting quantum state before being returned as binary strings. Due to quantum device noise, some sampled configurations may be physically invalid, failing to conserve electron number or spin. HI-VQE addresses this using the configuration recovery process from the [qiskit-addon-sqd](/docs/addons/qiskit-addon-sqd) package, so that users can either correct invalid configurations or discard them.

The valid configurations then undergo an optional screening step to remove those predicted to contribute minimally. This reduces the dimension of the subspace, thereby lowering the cost of the diagonalization step. If the screening is enabled, then a preliminary subspace Hamiltonian is constructed from the valid configurations and a diagonalization is performed with very loose termination critieria. Though the accuracy of the resulting amplitudes for each configuration is low, it is effective for predicting which configurations to leave out of the subspace this iteration, and it is fast to compute.

The selected configurations are added to the subspace, and the system's Hamiltonian is projected into this subspace. The subspace updates iteratively, preserving the most relevant configurations across iterations. This approach contrasts with alternative methods because the quantum circuit does not need to approximate the full ground state at each step.

Next, the subspace Hamiltonian is classically diagonalized to obtain the lowest eigenvalue and its corresponding eigenvector, representing an approximation of the ground state and its energy. As the subspace quality improves through iterations, the computed ground state better approximates the true ground state. An additional screening step can be performed at this point to remove any configurations from the subspace that don't have a substantial contribution to the computed ground state. This step ensures that the subspace carried into the next iteration is as compact as possible. This is evaluated based on the amplitudes that are returned by the diagonalization, as these represent each configuration's importance contribution to the computed ground state.

A convergence check then determines whether further training would improve results. If so, then an optional classical expansion step is performed, the quantum circuit parameters are updated to further minimize the computed energy, and the process repeats. The classical expansion step generates additional configurations for the subspace, supplementing the configurations sampled from the quantum device. It first identifies the configuration with the largest amplitude in the diagonalization results, before generating new configurations with single and double excitations from the identified configuration. The desired number of these configurations are then added to the subspace.

Once it is determined that the iterations have converged, HI-VQE returns the computed ground state (in the form of the states in the subspace and their amplitudes in the ground state wave function), its energy, and an energy variance measure that gives an indication of whether the computed state forms an eigenstate of the system's Hamiltonian.

Users are able to decide the quantum circuit used and the number of shots taken for each quantum circuit, as well as control the subspace size or enable the classical generation of additional configurations to assist the quantum generated configurations. In this way users can tailor the behavior of HI-VQE to suit their desired applications.

## Get started

Runs are billed per execution at the function size you choose, with no qubit limit; see [Limits and billing](#limits-and-billing). If you hold or need a license, see [Licensed use](#licensed-use).

The instance you authenticate with needs [Qiskit Functions access enabled](/docs/guides/access-instances-platform-apis#configure-qiskit-functions-access-on-an-instance), or the function will not appear in the catalog.

Save your [IBM Quantum&reg; API key](http://quantum.cloud.ibm.com/) [to your local environment](/docs/guides/functions-get-started#install-qiskit-functions-catalog-client) if you have not already, then select the Qiskit Function as follows:

In [1]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [ ]:
# Load the function
function = catalog.load("qunova/hivqe-chemistry")

## Example

The first example shows how to compute the ground state energy for an NH3 molecule using the HI-VQE algorithm.

#### Define the molecular geometry and options

The molecular geometry of NH3 is provided with Cartesian coordinates separated with ";" for each atom.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Additional options can be defined and provided for molecular system in the following dictionary format.
For every option you can set, with its type, default and valid range, see
[`molecule_options`](/docs/api/functions/qunova-chemistry#molecule_options) and
[`hivqe_options`](/docs/api/functions/qunova-chemistry#hivqe_options) in the API reference.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Before running, you can check the arguments with `validate_arguments()`. It
checks their names, types and ranges against the function's argument schema
without creating a job, so it costs nothing. It returns `{"valid": True}`, or
raises an exception naming the argument. `run()` makes the same check before
it creates a job, so a call that fails it is refused straight away and is not
billed. `function.arguments_schema` shows every argument the function accepts,
with its type, range and default.

This check does not cover the function size. To check that the size fits the
problem, use the [size estimator](#size-estimator).

In [ ]:
function.validate_arguments(
    {
        "geometry": geometry,
        "backend_name": backend_name,
        "max_states": 2000,
        "max_expansion_states": 10,
        "molecule_options": molecule_options,
        "hivqe_options": hivqe_options,
    }
)

Execute the function with geometry and option inputs. This example uses 16
qubits, from the 8 orbitals of the STO-3G basis for NH3, so it runs at the
default function size `"m"`.

In [ ]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is a backend you have access to,
    # for example, "ibm_marrakesh".
    backend_name=backend_name,
    # `function_size` sets the resources the job runs on and what it
    # costs. See "Limits and billing" below.
    function_size="m",
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

It is a good idea to print the Function job ID so that it can be provided in support requests if something goes wrong.

In [6]:
print("Job ID:", job.job_id)

Job ID: e5ced6f2-fd1d-4244-a6aa-bd27cfb0cdee


Check your Qiskit Function workload's [status](/docs/guides/functions-get-started#check-job-status) or return [results](/docs/guides/functions-get-started#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


After the job is completed, the results can be obtained with `result()` instance.

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'energy': -55.52108557170985, 'energy_history': [-55.51901898989887, -55.52056881448526, -55.52065046778772, -55.520690696813716, -55.520691108428, -55.520708448092634, ...], 'energy_variance': 3.066239097617371e-10, 'results_file_name': '20260914_105...vqe_result.h5'}


To access the ground state energy, use the `"energy"` key. The
`"results_file_name"` key provides the name of the h5 results file that can
be downloaded using `QiskitServerless`.

The response also carries:
- `"energy_variance"`: the energy variance of the final trial state, as a
  single number,
- `"energy_history"`: the energies computed along the way, two per
  iteration,
- `"function_version"`: the version of the function that ran the job, which
  is worth quoting in a support request, and
- `"metadata"`: a `"resource_usage"` dictionary giving the time spent in
  each stage of the job, including QPU time.

The results file contains:
- `"final_energy"`: the energy of the final trial state,
- `"final_energy_variance"`: the energy variance of the final trial state,
- `"energy_history"`: the energy history over HI-VQE's training,
- `"eigenvector"`: the CI coefficients of the final trial state's wave function, and
- `"states"`: the corresponding electron configurations comprising the final trial state wave function.

In [ ]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: "
    f"{abs(fci_energy - hivqe_energy) * 1000} mHa"
)

|Exact Energy - HI-VQE Energy|: 0.06246299427914437 mHa


To download the results file for the calculation, you will need to create a `QiskitServerless` instance using the same credentials as `QiskitFunctionCatalog`.

In [ ]:
from qiskit_ibm_catalog import QiskitServerless

serverless = QiskitServerless(
    channel="ibm_quantum_platform",
    instance="INSTANCE_CRN",
    token="YOUR_API_KEY",
)

The file can then be downloaded to the current directory and opened as shown below. Reading it requires the `h5py` package, which you can install with `pip install h5py`.

In [ ]:
import h5py

# Save the file in the current directory under its own name
results_file_name = result["results_file_name"]
serverless.file_download(
    results_file_name, function, target_name=results_file_name
)

# Read data from the results file for a single calculation
with h5py.File(results_file_name, "r") as f:
    states = f["states"].asstr()[:]
    eigenvector = f["eigenvector"][:]
    energy_history = f["energy_history"][:]
    final_energy = f["final_energy"][0]
    final_energy_variance = f["final_energy_variance"][0]

#### Run from an FCIDUMP file

Instead of a geometry, you can give the problem as an FCIDUMP file of its
integrals, for example one written by PySCF or by another quantum chemistry
program. Upload the file to the function, then pass its name as
`fcidump_file_name`:

- The integrals are used as they are. No mean-field calculation is run, and
  `geometry` and `molecule_options` are ignored if you pass them as well.
- Every orbital in the file is active, so the problem uses 2 × `NORB` qubits.
- The file can be plain text or gzip-compressed. Only closed-shell singlets
  are supported: `MS2=0` and an even `NELEC`.
- The first `NELEC`/2 orbitals are taken as the occupied orbitals of the
  reference state. If the orbitals of the file are not in order of energy,
  the job log says so, as the run can converge slowly from that state.

This writes the NH3 problem of the example to a file with PySCF:

In [ ]:
from pyscf import gto, scf
from pyscf.tools import fcidump

mol = gto.M(atom=geometry, basis="sto3g")
fcidump.from_scf(scf.RHF(mol).run(), "nh3.FCIDUMP")

Upload the file and run. Pass the file name alone, without a directory. The
file stays uploaded until you remove it with `serverless.file_delete()`.

In [ ]:
serverless.file_upload("nh3.FCIDUMP", function)

job = function.run(
    fcidump_file_name="nh3.FCIDUMP",
    backend_name=backend_name,
    function_size="m",
    max_states=2000,
    max_expansion_states=10,
    hivqe_options=hivqe_options,
)

## Limits and billing

What applies to your runs depends on whether you hold a license token.

|  | Without a token | With a license token |
| --- | --- | --- |
| **Qubits** | No limit | What your license grants |
| **What you pay** | Per run, at the function size you choose | Per your license agreement |

<Admonition type="note">
  Trial access is limited to 20 qubits. A job that needs more fails with error code 4500.
</Admonition>

Every run is submitted at a function size and is checked against it, with or
without a token. The two sections that follow explain how to choose it.

### Function sizes

`function_size` tells the platform what resources to run the job on, and it
is what the run is billed at. Pass it to `run()` alongside the function's own
arguments.

| `function_size` | Qubits | License fee per run |
| --- | --- | --- |
| `"s"` | Up to 10 | 0.4 RU |
| `"m"` | Up to 20 | 1.0 RU |
| `"l"` | No limit | 4.0 RU |

Fees are given in resource units. See [IBM Quantum
plans](/docs/guides/plans-overview) for what a resource unit costs on your
plan.

If you leave `function_size` out, the job runs at `"m"`.

If the size does not match the problem:

- **Too small:** the run is rejected before any QPU time is spent, with error
  code 1221 naming the size to use. This applies to runs with a license token
  too. Without a token, the rejected run is still billed at the size you
  chose, because a run is billed once it starts.
- **Too large:** the run goes ahead and is billed at the larger size, with a
  warning naming the size that would have been enough.

To get the size right the first time, check it before you submit, as the next
section shows.

### Check the size before you run

The size a run needs depends only on how many qubits the problem uses, which
follows from your `molecule_options`:

| Your `molecule_options` | Qubits |
| --- | --- |
| `avas_aolabels` set | 2 × the number of orbitals AVAS selects (threshold `avas_threshold`, default 0.2) |
| `active_orbitals` set | 2 × the number of active orbitals |
| Neither | 2 × the number of atomic orbitals in the basis |

With [`fcidump_file_name`](#run-from-an-fcidump-file), the size depends only on
the file: 2 × `NORB`, the number of orbitals in its header. `molecule_options`
are then ignored.

Submit at the smallest size that fits: up to 10 qubits is `"s"`, up to 20 is
`"m"`, and anything larger is `"l"`. The NH3 example has 8 orbitals in STO-3G,
so it needs 16 qubits and runs at `"m"`, as in the [Example](#example) above.

To find the size on your own machine before you submit, use the
[size estimator](#size-estimator). It gives the size IBM and the function
check your run at, needs only PySCF, and costs nothing.

## Licensed use

A license grants a qubit limit of its own. Runs made with a license token are
still submitted at a function size and checked against it; see
[Function sizes](#function-sizes).

To request access and a license, fill in [this form](https://forms.office.com/r/zN3hvMTqJ1), or email
[qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com) with your inquiry.

### Example of licensed function use

Licensed users are issued a license token, and then must use a wrapper library to submit their license token to the function.
The wrapper library can be installed [from PyPI](https://pypi.org/project/hivqe-qiskit-function-utils/) with `pip install hivqe-qiskit-function-utils`.
The below example shows how this library should be used to submit your license token when calling the function.

In [ ]:
import math
from qiskit_ibm_catalog import QiskitFunctionsCatalog
from hivqe_qiskit_function_utils import FunctionWrapper

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

molecule_geometry = f"""
O 0 0 0;
H {-0.957*math.sin(math.radians(104.5)/2.0)} {0.957*math.cos(math.radians(104.5)/2.0)} 0;
H {0.957*math.sin(math.radians(104.5)/2.0)} {0.957*math.cos(math.radians(104.5)/2.0)} 0
"""

hivqe = FunctionWrapper(
    token="your_hivqe_license_token",
    function=catalog.load("qunova/hivqe-chemistry"),
)
job = hivqe.run(
    geometry=molecule_geometry,
    backend_name=backend_name,  # a backend you have access to
    # 14 qubits fit the default "m"; a problem over 20 qubits needs "l"
    function_size="m",
    max_states=10000,
    max_expansion_states=1000,
    hivqe_options={"ansatz": "epa", "max_iter": 10},
)
result = job.result()

## Performance

This section shows the demonstrated benchmark calculations of HI-VQE with a 24-qubit case for Li2S, a 40-qubit case for an N2 molecule, and a 44-qubit case for an FeP-NO system.

#### Dissociation potential energy surface curve for an Li2S molecule with 24 qubits

The PES curve is shown with the FCI reference and initial guess from RHF, together with the energy error from the FCI reference.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the Li2S system](/docs/images/guides/qunova-chemistry/Li2S_PES.avif).

The calculations have been conducted with the following geometries and options.

In [10]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [11]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # a backend you have access to
        function_size="l",  # 38 qubits, more than "m" allows
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

The red dots represent the HI-VQE calculation results for six different geometries, and three geometries corresponding to 1.51, 2.40, and 3.80 Angstrom are provided as input in the above cell.

#### Dissociation PES curve for an N2 molecule with 40 qubits

The nitrogen molecule has been identified as a multireference system with large correlation energy contributions beyond the Hartree-Fock state. We conducted a benchmark calculation for the N2 molecule with cc-pvdz basis, (20o,14e) using the homo-lumo active orbital selection. The complete active space (CAS) number to represent this problem is 6,009,350,400. It is not possible to obtain the eigenvalue problem solution (for energy and electronic structure) with this number of states using a powerful desktop (16cpu/64GB). With HI-VQE, users can efficiently search the subspace of CAS states to find chemically accurate results while saving computation resources significantly. The following plots show the PES curve of 40 qubits HI-VQE calculation of the N2 molecule dissociation.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the N2 system](/docs/images/guides/qunova-chemistry/N2_PES_40qubits.avif)

#### Dissociation PES curve for five-coordinated iron(II)-porphyrin with an NO system with 44 qubits

Another interesting chemical system is an iron(II)-porphyrin (FeP) complex with a coordinated nitric oxide (NO) ligand, which represents a biologically relevant metalloporphyrin system that plays crucial roles in various physiological processes. In this example, HI-VQE has been utilized to estimate the accurate potential energy surface curve of the intermolecular interaction between FeP and NO (ground state energy for differently separated geometries). The combined system has 450 orbitals and 202 electrons (450o,202e) with 6-31g(d) basis in total. The homo-lumo active orbital selection was utilized to calculate the smaller case from the real case with (22o,22e). From the following benchmark results, we were able to achieve the chemical accuracy (>&nbsp;1.6 mHa) with a state-of-the-art classical computer chemistry calculation of CASCI(DMRG) (22o,22e) reference.

![Image showing that HI-VQE produces solutions within chemical accuracy of a classical reference PES curve for the FeP-NO system](/docs/images/guides/qunova-chemistry/fepno_44qubits.avif)

## Benchmarks

- Exact matrix size is the number of determinants for exact solution, such as FCI and CASCI.
- HI-VQE calculation samples and calculates the subspace of it (as in, HI-VQE matrix size).
- Total time includes QPU runtime and Qiskit Function runs with CPU.
- Accuracy is estimated from the energy difference from exact solution.

|   Chemical system  | Number of qubits | Exact matrix size |  HI-VQE matrix size  | E(diff) from exact (mHa) | Number of iteration | Total time    | QPU runtime usage |
| ------------------ | ---------------- | ----------------- | ------------------- | ------------------------ | ------------------- | ------------- | ----------------- |
|  $NH_3$ (8o,10e)   |  16              |  3136             |  1936               |  0.08                    |  6                  | 37&nbsp;s     | 34&nbsp;s         |
|  $Li_2S$ (10o,10e) |  20              |  63504            |  3969               |  0.60                    |  5                  | 250&nbsp;s    | 50&nbsp;s         |
|  $NH_3$ (15o,10e)  |  30              |  9018009          |  49729              |  0.90                    |  5                  | 354&nbsp;s    | 54&nbsp;s         |
|  $N_2$ (16o,14e)   |  32              |  130873600        |  1798281            |  1.10                    |  9                  | 6531&nbsp;s   | 121&nbsp;s        |
|  $3H_2O$ (18o,24e) |  36              |  344622096        |  399424             |  0.90                    |  24                 | 5174&nbsp;s   | 130&nbsp;s        |
|  $N_2$ (20o,14e)   |  40              |  6009350400       |  9012004            |  1.20                    |  21                 | 46547&nbsp;s  | 258&nbsp;s        |

## Fetch error messages

If your workload fails, the status will be `ERROR` and calling
`job.result()` will raise an exception:

In [12]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: ["runner.UnknownRuntimeError: 'An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance. -- https://docs.quantum.ibm.com/errors#1500'\n"]

In [13]:
job.status()

'ERROR'

Every failure reports an error code, a message describing what went wrong,
and a solution saying what to do about it. The codes follow [IBM's error
code registry](/docs/errors), and the ones this function returns are:

| Code | What happened |
| --- | --- |
| 1007 | The `backend_name` you gave does not exist, or is not available to you |
| 1201 | A license token was supplied but no matching license was found |
| 1221 | An argument did not validate, or `function_size` was too small for the problem. The message names the size to use |
| 1500 | Something unexpected went wrong. Quote the job ID when you report it |
| 2409 | The license has expired, has been revoked, or was used from an instance it was not issued for |
| 4500 | The problem needs more qubits than your license or trial access allows |
| 4630 | Your license could not be read. Nothing is wrong with what you submitted; contact us with the job ID |
| 4631 | A Qiskit Runtime job made by the run failed or was cancelled, and Runtime gave no error code of its own. Submit the job again |

When a Qiskit Runtime job made by the run fails with an error code of its
own, such as 1305, that code is reported instead; look it up in the
registry.

Several codes cover more than one cause, so read the message as well as the
code. If you need help with a failure, send the Function job ID to
[qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com).

A call with a wrong argument name, type or range does not get this far:
`run()` refuses it before a job is created, as described in
[Example](#example).

## Size estimator

This is the code IBM and the function use to size a job, so the size it gives
is the one your run is placed and checked at. It needs only PySCF
(`pip install pyscf`), submits nothing, and costs nothing. Run the cell below,
or save it as `hivqe_chemistry_size_estimator.py`. You only need to call
`estimate_size`; the rest of the file is how the size is worked out.

In [ ]:
"""Size estimator for Qunova HI-VQE Chemistry. Requires PySCF only.

estimate_size(geometry, molecule_options=...) returns "s", "m" or "l" from the
number of qubits the problem needs: two per active orbital. The active space is
the orbitals AVAS selects if avas_aolabels is set, else active_orbitals if set,
else every orbital in the basis. With fcidump_file_name, the path of an FCIDUMP
file, it is every orbital of the file, and geometry and molecule_options are
ignored, as the function ignores them. Other inputs are accepted and ignored.
"""
from __future__ import annotations

import gzip
import re
from typing import Any, Literal

from pyscf import gto, scf
from pyscf.mcscf import avas

Size = Literal["s", "m", "l"]
_ORDER = {"s": 0, "m": 1, "l": 2}

# Upper bounds (inclusive) on the number of qubits for each size.
_SMALL = 10
_MEDIUM = 20

# The function's defaults for the molecule_options that decide the active space.
_DEFAULTS = {
    "active_orbitals": None,
    "avas_aolabels": None,
    "avas_threshold": 0.2,
    "basis": "sto-3g",
    "charge": 0,
    "symmetry": False,
    "symmetry_subgroup": None,
    "unit": "angstrom",
    "nucmod": None,
    "pseudo": None,
    "cart": False,
    "magmom": None,
}

# AVAS weights below this are numerical noise; counting them would make the size vary
# from one call to the next.
_AVAS_WEIGHT_TOLERANCE = 1e-10


def estimate_size(problem: Any, **kwargs: Any) -> Size:
    """Return the size ("s", "m" or "l") required to run the function."""
    metrics = _measure(problem, **kwargs)
    if _is_small(metrics):
        return "s"
    if _is_medium(metrics):
        return "m"
    return "l"


def _measure(problem: Any, **kwargs: Any) -> dict:
    """Return the number of qubits the run uses for this geometry and these options."""
    if kwargs.get("fcidump_file_name"):
        return {"num_qubits": 2 * _fcidump_num_orbitals(kwargs["fcidump_file_name"])}
    options = {**_DEFAULTS, **(kwargs.get("molecule_options") or {})}
    mol = gto.M(
        atom=problem,
        basis=options["basis"],
        spin=0,  # the function runs singlets only
        charge=options["charge"],
        symmetry=options["symmetry"],
        symmetry_subgroup=options["symmetry_subgroup"],
        unit=options["unit"],
        nucmod=options["nucmod"],
        pseudo=options["pseudo"],
        cart=options["cart"],
        magmom=options["magmom"],
        verbose=0,
    )

    if options["avas_aolabels"] is None:
        # The active space is the orbitals given, or the whole basis; building the molecule
        # is enough to know it.
        if options["active_orbitals"]:
            return {"num_qubits": 2 * len(options["active_orbitals"])}
        return {"num_qubits": 2 * mol.nao_nr()}

    # AVAS picks the active space from an RHF, so it has to run one.
    rhf = scf.RHF(mol)
    rhf.verbose = 0
    rhf.kernel()
    selection = avas.AVAS(
        mf=rhf,
        aolabels=options["avas_aolabels"],
        threshold=max(options["avas_threshold"], _AVAS_WEIGHT_TOLERANCE),
    )
    selection.verbose = 0
    selection.kernel()
    return {"num_qubits": 2 * selection.ncas}


# An FCIDUMP header is read up to its end (&END or /). It is a few lines; its
# only part that grows with the problem is ORBSYM, a few characters per orbital.
_FCIDUMP_MAX_HEADER_LINES = 1000


def _fcidump_num_orbitals(path: str) -> int:
    """NORB from the header of an FCIDUMP file, plain or gzip-compressed,
    without reading the integrals."""
    with open(path, "rb") as raw:
        gzipped = raw.read(2) == b"\x1f\x8b"
    opener = gzip.open if gzipped else open
    header = []
    try:
        with opener(path, "rt") as in_file:
            for line_number, line in enumerate(in_file):
                header.append(line.upper())
                if "&END" in header[-1] or "/" in line:
                    break
                if line_number >= _FCIDUMP_MAX_HEADER_LINES:
                    raise ValueError("No end of header (&END or /) was found in the FCIDUMP file.")
            else:
                raise ValueError("No end of header (&END or /) was found in the FCIDUMP file.")
    except (UnicodeDecodeError, OSError, EOFError) as error:
        raise ValueError(
            "The FCIDUMP file could not be read as text or as gzip-compressed text."
        ) from error
    match = re.search(r"NORB\s*=\s*(\d+)", "".join(header))
    if match is None or int(match.group(1)) < 1:
        raise ValueError("No valid NORB entry was found in the FCIDUMP file header.")
    return int(match.group(1))


def _is_small(metrics: dict) -> bool:
    """Determine if the metrics fit T-size 's'."""
    return metrics["num_qubits"] <= _SMALL


def _is_medium(metrics: dict) -> bool:
    """Determine if the metrics fit T-size 'm'."""
    return metrics["num_qubits"] <= _MEDIUM


def validate_size(provided_size: Size, problem: Any, **kwargs: Any) -> None:
    """Call at the start of the function. Raise if ``provided_size`` is smaller than
    the estimated size; the estimated size or a larger one is accepted."""
    expected = estimate_size(problem, **kwargs)
    if _ORDER[provided_size] < _ORDER[expected]:
        raise ValueError(
            f"Size '{provided_size}' is too small for this problem; expected '{expected}' or larger."
        )

For the NH3 molecule in the [Example](#example):

In [ ]:
# NH3 in STO-3G has 8 orbitals, so 16 qubits: "m"
print(estimate_size(geometry, molecule_options=molecule_options))

For an FCIDUMP file, pass its path as `fcidump_file_name`, as you will pass its
name to `run()`:

In [ ]:
# nh3.FCIDUMP, written in "Run from an FCIDUMP file", has 8 orbitals: "m"
print(estimate_size(None, fcidump_file_name="nh3.FCIDUMP"))

## Changelog

### 5.0.4

- Runs can take the problem from an FCIDUMP file you upload, in place of a geometry: pass its name as `fcidump_file_name`. The size estimator accepts it too. See [Run from an FCIDUMP file](#run-from-an-fcidump-file).
- A run whose Qiskit Runtime job fails or is cancelled now reports Runtime's own error code, or 4631 when Runtime gives none, with a message and solution. It used to report an unexpected error, and the result lost its `"resource_usage"`.
- A failed run reports its error once. It used to be reported twice.
- `"resource_usage"` lists every stage of the job, with zero for a stage the run did not reach.

### 5.0.3

- Runs made with a license token, and trial runs, are now held to the function size like other runs: one submitted at a size too small for the problem is rejected with error code 1221. The default `"m"` fits up to 20 qubits.
- Added a size estimator to run on your own machine at no charge. It is the code IBM and the function use to size a job. See [Size estimator](#size-estimator).
- Arguments are checked against the function's argument schema when you call `run()`. A malformed call is refused before a job is created, and is not billed. Check arguments in advance with `validate_arguments()`. See [Example](#example).
- Runs that pass `orbital_coeffs` no longer fail. It also accepts a NumPy array, such as PySCF's `mo_coeff`.
- `ansatz_entanglement` accepts any letter case, as `ansatz` and `noons_level` already did.
- `avas_threshold` accepts a whole number, such as `0`.
- `active_orbitals`, `frozen_orbitals` and `avas_aolabels` must not repeat an entry.

### 5.0.0

- Runs without a license token are no longer limited to 20 qubits. They are billed per run at a function size you choose with the new `function_size` argument: `"s"` (up to 10 qubits), `"m"` (up to 20, the default) or `"l"` (no limit). A size too small for the problem is rejected with error code 1221 before any QPU time is spent, but is still billed at the size chosen. See [Limits and billing](#limits-and-billing).
- The result now includes `"function_version"`, the version of the function that ran the job, and `"metadata"`, whose `"resource_usage"` gives the time spent in each stage of the job, including QPU time.
- Errors use the error codes listed in [Fetch error messages](#fetch-error-messages), and each one says what to do about it.

### 4.0.0

- The result no longer includes `"states"` or `"eigenvector"`. They are now saved in an HDF5 results file, together with `"final_energy"`, `"final_energy_variance"`, and `"energy_history"`.
- Added `"results_file_name"` to the result. It gives the name of the results file, which you can download with `serverless.file_download()`. Reading the file requires the `h5py` package. See the [Example](#example) section for how to download and read it.

## Get support

You can send an email to [qiskit.support@qunovacomputing.com](mailto:qiskit.support@qunovacomputing.com) for help with this function.

If you want help with troubleshooting a specific error, please provide the Function job ID of the job that encountered the error.

## Next steps

<Admonition type="tip" title="Recommendations">
- Request access and a license, if you need one, by filling in [this form](https://forms.office.com/r/zN3hvMTqJ1).
- Visit the [API reference](/docs/api/functions/qunova-chemistry) for this Qiskit Function.
- Review [Pellow-Jarman, A., et al. (2025).  HIVQE: Handover Iterative Variational Quantum Eigensolver for Efficient Quantum Chemistry Calculations. arXiv preprint arXiv:2503.06292](https://arxiv.org/abs/2503.06292).
- Try the [Dissociation PES curves with Qunova HiVQE](/docs/tutorials/qunova-hivqe) tutorial.
</Admonition>